# Paper 1 schema probe (not a test)
Reads only the declared burned days in `SCHEMA_PROBE.md`. Runs no estimator. Writes `schema_probe_results.zip` for the author to return.

In [ ]:
import io, json, time, zipfile, datetime, hashlib, requests
import pandas as pd
OUT = {}
LOG = []
H = {"User-Agent": "Mozilla/5.0 (research schema probe)"}
def get(name, url, **kw):
    t = datetime.datetime.utcnow().isoformat() + "Z"
    try:
        r = requests.get(url, headers=H, timeout=60, **kw)
        LOG.append({"name": name, "url": url, "status": r.status_code, "bytes": len(r.content), "utc": t,
                    "sha256": hashlib.sha256(r.content).hexdigest()})
        OUT[name + ".raw"] = r.content
        return r
    except Exception as e:
        LOG.append({"name": name, "url": url, "error": repr(e), "utc": t})
        return None
def schema(name, df):
    OUT[name + ".schema.json"] = json.dumps({
        "rows": int(len(df)), "columns": [str(c) for c in df.columns],
        "dtypes": {str(c): str(t) for c, t in df.dtypes.items()},
        "head": df.head(3).astype(str).to_dict(orient="records")}, ensure_ascii=False, indent=1).encode()

In [ ]:
# TWSE T86: three major institutional investors, per stock, two burned dates
for d in ["20240102", "20240103"]:
    r = get(f"twse_T86_{d}", f"https://www.twse.com.tw/rwd/zh/fund/T86?date={d}&selectType=ALLBUT0999&response=json")
    if r is not None and r.ok:
        try:
            j = r.json(); df = pd.DataFrame(j.get("data", []), columns=j.get("fields"))
            schema(f"twse_T86_{d}", df)
        except Exception as e: LOG.append({"name": f"twse_T86_{d}", "parse_error": repr(e)})
    time.sleep(3)

In [ ]:
# TPEx: three major institutional investors, per stock (ROC date format 113/01/02)
for d in ["113/01/02", "113/01/03"]:
    tag = d.replace("/", "")
    r = get(f"tpex_3insti_{tag}", "https://www.tpex.org.tw/web/stock/3insti/daily_trade/3itrade_hedge_result.php",
            params={"l": "zh-tw", "o": "json", "se": "EW", "t": "D", "d": d})
    if r is not None and r.ok:
        try:
            j = r.json(); rows = j.get("aaData") or (j.get("tables") or [{}])[0].get("data", [])
            schema(f"tpex_3insti_{tag}", pd.DataFrame(rows))
        except Exception as e: LOG.append({"name": f"tpex_3insti_{tag}", "parse_error": repr(e)})
    time.sleep(3)

In [ ]:
# JPX weekly trading by type of investors: list the page's files, fetch only the first-week-January-2024 file
r = get("jpx_investor_type_page", "https://www.jpx.co.jp/english/markets/statistics-equities/investor-type/index.html")
import re
links = re.findall(r'href="([^"]+\.xlsx?)"', r.text) if (r is not None and r.ok) else []
OUT["jpx_links.json"] = json.dumps(links, indent=1).encode()
cand = [l for l in links if "2401" in l or "240105" in l or "2024" in l][:1]
for l in cand:
    url = l if l.startswith("http") else "https://www.jpx.co.jp" + l
    rr = get("jpx_week_2024_01", url)
    if rr is not None and rr.ok:
        try:
            x = pd.read_excel(io.BytesIO(rr.content), sheet_name=None, header=None)
            for s, df in list(x.items())[:3]: schema(f"jpx_week_2024_01_{s}", df)
        except Exception as e: LOG.append({"name": "jpx_week_2024_01", "parse_error": repr(e)})
print("JPX candidate files:", cand, "of", len(links))

In [ ]:
# NSE participant-wise F&O trading volume, 2 Jan 2024 (archive CSV; may need a session cookie)
s = requests.Session(); s.headers.update(H)
try: s.get("https://www.nseindia.com", timeout=30)
except Exception as e: LOG.append({"name": "nse_home", "error": repr(e)})
url = "https://archives.nseindia.com/content/nsccl/fao_participant_vol_02012024.csv"
try:
    r = s.get(url, timeout=60)
    LOG.append({"name": "nse_participant_vol_20240102", "url": url, "status": r.status_code, "bytes": len(r.content)})
    OUT["nse_participant_vol_20240102.raw"] = r.content
    if r.ok: schema("nse_participant_vol_20240102", pd.read_csv(io.BytesIO(r.content), skiprows=1))
except Exception as e: LOG.append({"name": "nse_participant_vol_20240102", "error": repr(e)})

In [ ]:
OUT["log.json"] = json.dumps(LOG, indent=1).encode()
with zipfile.ZipFile("schema_probe_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for k, v in OUT.items(): z.writestr(k, v)
for e in LOG: print(e.get("name"), e.get("status", e.get("error", e.get("parse_error"))))
try:
    from google.colab import files; files.download("schema_probe_results.zip")
except Exception: print("Saved schema_probe_results.zip")